In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import urllib.request
import random
from pathlib import Path

In [9]:
DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"
RAW_DIR = DATA_DIR / "raw"
STRUCTURES_DIR = DATA_DIR / "structures"
for d in (SPLITS_DIR, RAW_DIR, STRUCTURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [29]:
url = "https://cdn.rcsb.org/resources/sequence/clusters/clusters-by-entity-50.txt"
clusters_file = RAW_DIR / "clusters-by-entity-50.txt"
urllib.request.urlretrieve(url, clusters_file)

(PosixPath('../data/raw/clusters-by-entity-50.txt'),
 <http.client.HTTPMessage at 0x7f462e1a6c60>)

In [30]:
random.seed(42)

valid_clusters = []

with open(clusters_file, "r") as f:
    for cluster_idx, line in enumerate(f):
        members = line.strip().split()
        af_members = [m for m in members if m.startswith("AF_")]
        if len(af_members) >= 10:
            valid_clusters.append((cluster_idx, af_members))

random.shuffle(valid_clusters)

In [4]:
SPLIT_CLUSTERS = {
    "train": 400,
    "eval": 50,
    "test": 50,
}  # x10 members = 4000/500/500 proteins
MEMBERS_PER_CLUSTER = 10

In [ ]:
selected_clusters = valid_clusters[: sum(SPLIT_CLUSTERS.values())]

In [32]:
# one split label per selected cluster, in order: 400x train, 50x eval, 50x test
split_labels = [
    label for label, amount in SPLIT_CLUSTERS.items() for _ in range(amount)
]
assert len(split_labels) == len(selected_clusters)

df = pd.DataFrame(
    {"protein_id": pid, "cluster_id": cluster_id, "split": split}
    for split, (cluster_id, members) in zip(split_labels, selected_clusters)
    for pid in random.sample(members, MEMBERS_PER_CLUSTER)
)

In [33]:
for split_name in SPLIT_CLUSTERS:
    split_ids = df[df["split"] == split_name]["protein_id"]

    split_ids.to_csv(SPLITS_DIR / f"{split_name}_af.txt", index=False, header=False)

In [5]:
import re
import subprocess


def expected_af_files(ids_file):
    """Structure file names that download_structures.sh should produce for a split file."""
    names = set()
    for line in open(ids_file):
        m = re.match(r"AF_AF(.+?)(F\d+)_\d+$", line.strip())
        if m:
            names.add(f"AF-{m.group(1)}-{m.group(2)}-model_v6.cif")
    return names


def check_structures(structures_dir="structures", splits=SPLIT_CLUSTERS, verbose=True):
    """Compare downloaded files with the split ID files; return {split: set of missing file names}."""
    missing_by_split = {}
    for split_name in splits:
        expected = expected_af_files(SPLITS_DIR / f"{split_name}_af.txt")
        out_dir = DATA_DIR / structures_dir / split_name / "af"
        present = {p.name for p in out_dir.glob("*.cif")} if out_dir.exists() else set()
        missing_by_split[split_name] = expected - present
        if verbose:
            status = "OK     " if not missing_by_split[split_name] else "MISSING"
            print(
                f"{status} {structures_dir}/{split_name}/af: {len(expected & present)}/{len(expected)} files"
            )
    return missing_by_split

In [35]:
%%writefile ../scripts/download_structures.sh
#!/usr/bin/env bash
# Usage: ./download_structures.sh <data_dir> train af
# Downloads sequentially with a short pause and retries (the servers answer
# "429 Too Many Requests" to bursts). Existing files are skipped, so it is safe to re-run.
DATA_DIR=$1  # e.g. ../data
SPLIT=$2     # train / eval / test
DB=$3        # pdb / af

OUT="${DATA_DIR}/structures/${SPLIT}/${DB}"
IDS="${DATA_DIR}/splits/${SPLIT}_${DB}.txt"
mkdir -p "$OUT"

download() {  # download <url>; retries with growing backoff
    local url=$1 dest="$OUT/$(basename "$1")"
    [[ -s "$dest" ]] && return 0
    for attempt in 1 2 3 4 5 6; do
        if curl -sf -A "Wget/1.21" -o "$dest.part" "$url"; then
            mv "$dest.part" "$dest"
            sleep 0.2
            return 0
        fi
        sleep $((attempt * 10))
    done
    rm -f "$dest.part"
    echo "FAILED: $url" >&2
    return 1
}

if [[ "$DB" == "pdb" ]]; then
    urls=$(awk -F'_' '{print "https://files.rcsb.org/download/"$1".cif"}' "$IDS" | sort -u)
elif [[ "$DB" == "af" ]]; then
    urls=$(python3 -c "
import re
for line in open('${IDS}'):
    m = re.match(r'AF_AF(.+?)(F\d+)_\d+$', line.strip())
    if m:
        print(f'https://alphafold.ebi.ac.uk/files/AF-{m.group(1)}-{m.group(2)}-model_v6.cif')
" | sort -u)
fi

failed=0
for url in $urls; do
    download "$url" || failed=$((failed + 1))
done
echo "${SPLIT}/${DB}: done, ${failed} failed"


Overwriting ../scripts/download_structures.sh


In [ ]:
!chmod +x ../scripts/download_structures.sh

In [11]:
subprocess.run(["../scripts/download_structures.sh", str(DATA_DIR), "eval", "af"])

CompletedProcess(args=['../scripts/download_structures.sh', '../data', 'eval', 'af'], returncode=0)

eval/af: done, 0 failed


In [14]:
subprocess.run(["../scripts/download_structures.sh", str(DATA_DIR), "test", "af"])

test/af: done, 0 failed


CompletedProcess(args=['../scripts/download_structures.sh', '../data', 'test', 'af'], returncode=0)

In [ ]:
subprocess.run(["../scripts/download_structures.sh", str(DATA_DIR), "train", "af"])

In [15]:
missing = check_structures("structures")

MISSING structures/train/af: 2410/4000 files
OK      structures/eval/af: 500/500 files
OK      structures/test/af: 500/500 files
